In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path.cwd().parent / 'data' / 'BigSolDBv2.0.csv'
OUT  = Path.cwd().parent / 'results' / 'audit_report.md'
print('Data file:', DATA)

Data file: /home/sg182/ML_Projects/BigSolDB-T/data/BigSolDBv2.0.csv


In [2]:
df = pd.read_csv(DATA)
expected = {
    'SMILES_Solute', 'Temperature_K', 'Solvent', 'SMILES_Solvent',
    'Solubility(mole_fraction)', 'Solubility(mol/L)', 'LogS(mol/L)',
    'Compound_Name', 'CAS', 'PubChem_CID', 'FDA_Approved', 'Source',
}
missing_cols = expected - set(df.columns)
extra_cols   = set(df.columns) - expected
n_rows = len(df)
print(f'rows: {n_rows:,}')
print(f'missing_cols: {sorted(missing_cols)}')
print(f'extra_cols:   {sorted(extra_cols)}')

rows: 103,944
missing_cols: []
extra_cols:   []


In [3]:
n_solutes  = df['SMILES_Solute'].nunique()
n_solvents = df['SMILES_Solvent'].nunique()
pair_key   = list(zip(df['SMILES_Solute'], df['SMILES_Solvent']))
n_pairs    = len(set(pair_key))
n_sources  = int(df['Source'].nunique()) if 'Source' in df.columns else None

print(f'unique solutes:  {n_solutes:,}')
print(f'unique solvents: {n_solvents:,}')
print(f'unique (solute, solvent) pairs: {n_pairs:,}')
print(f'unique source articles: {n_sources:,}')

if 'FDA_Approved' in df.columns:
    fda = df['FDA_Approved'].astype(str).str.lower().isin({'true','1','yes','y'}).sum()
    print(f'FDA-approved solute rows: {int(fda):,}')

unique solutes:  1,448
unique solvents: 209
unique (solute, solvent) pairs: 11,255
unique source articles: 1,595
FDA-approved solute rows: 24,483


In [4]:
null_counts = df.isna().sum().sort_values(ascending=False)
print('columns with missing values:')
print(null_counts[null_counts > 0].to_string())
print()

full_dup = int(df.duplicated().sum())
key_dup = int(df.duplicated(subset=['SMILES_Solute','SMILES_Solvent','Temperature_K']).sum())
print(f'full-row duplicates: {full_dup:,}')
print(f'duplicates on (SMILES_Solute, SMILES_Solvent, Temperature_K): {key_dup:,}')
print('  (these are legitimate replicate measurements defining the aleatoric floor)')

columns with missing values:
CAS                  7456
Solubility(mol/L)    2961
LogS(mol/L)          2961
PubChem_CID          2593



full-row duplicates: 0
duplicates on (SMILES_Solute, SMILES_Solvent, Temperature_K): 3,601
  (these are legitimate replicate measurements defining the aleatoric floor)


In [5]:
target = df['LogS(mol/L)']
print('LogS(mol/L) distribution')
print(target.describe().to_string())

T = df['Temperature_K']
print('\nTemperature_K distribution')
print(T.describe().to_string())
print(f'\nn_unique temperatures: {T.nunique():,}')

LogS(mol/L) distribution
count    100983.000000
mean         -1.029982
std           1.226869
min          -9.128883
25%          -1.770935
50%          -0.894021
75%          -0.150900
max           2.491071

Temperature_K distribution
count    103944.000000
mean        303.630294
std          15.749350
min         243.150000
25%         293.150000
50%         303.150000
75%         313.150000
max         425.770000

n_unique temperatures: 3,331


In [6]:
pair_group = df.groupby(['SMILES_Solute','SMILES_Solvent'])
n_meas_per_pair = pair_group.size()

thresholds = [1, 2, 3, 5, 10, 20]
for t in thresholds:
    print(f'pairs with >= {t:>3} measurements: {(n_meas_per_pair >= t).sum():,}')
print()
print(f'median measurements/pair: {int(n_meas_per_pair.median())}')
print(f'mean measurements/pair:   {n_meas_per_pair.mean():.2f}')
print(f'max measurements/pair:    {int(n_meas_per_pair.max())}')

pairs with >=   1 measurements: 11,255
pairs with >=   2 measurements: 10,851


pairs with >=   3 measurements: 10,831
pairs with >=   5 measurements: 10,730
pairs with >=  10 measurements: 4,448
pairs with >=  20 measurements: 226

median measurements/pair: 9
mean measurements/pair:   9.24
max measurements/pair:    50


In [7]:
T_min_per_pair = pair_group['Temperature_K'].min()
T_max_per_pair = pair_group['Temperature_K'].max()
delta_T_per_pair = T_max_per_pair - T_min_per_pair

print(f'pairs with only one T (ΔT=0): {int((delta_T_per_pair == 0).sum()):,}')
for t in [10, 20, 40, 60, 80]:
    print(f'pairs with ΔT >= {t:>2}K: {(delta_T_per_pair >= t).sum():,}')
print()
print(f'median ΔT among multi-T pairs: {delta_T_per_pair[delta_T_per_pair > 0].median():.1f} K')
print(f'max ΔT: {delta_T_per_pair.max():.1f} K')

pairs with only one T (ΔT=0): 409
pairs with ΔT >= 10K: 10,831
pairs with ΔT >= 20K: 10,676
pairs with ΔT >= 40K: 7,441
pairs with ΔT >= 60K: 238
pairs with ΔT >= 80K: 45

median ΔT among multi-T pairs: 40.0 K
max ΔT: 132.6 K


In [8]:
def pairs_meeting(min_meas, min_dT):
    ok = (n_meas_per_pair >= min_meas) & (delta_T_per_pair >= min_dT)
    return int(ok.sum())

for min_m, min_dT in [(3, 20), (5, 20), (5, 40), (10, 40), (10, 60)]:
    print(f'pairs with >= {min_m} meas AND ΔT >= {min_dT}K: '
          f'{pairs_meeting(min_m, min_dT):,}')

pairs with >= 3 meas AND ΔT >= 20K: 10,675
pairs with >= 5 meas AND ΔT >= 20K: 10,617
pairs with >= 5 meas AND ΔT >= 40K: 7,440
pairs with >= 10 meas AND ΔT >= 40K: 3,794
pairs with >= 10 meas AND ΔT >= 60K: 188


In [9]:
solvent_counts = df['SMILES_Solvent'].value_counts()
solute_counts  = df['SMILES_Solute'].value_counts()
print('top 15 solvents:')
print(solvent_counts.head(15).to_string())

top 15 solvents:
SMILES_Solvent
CCO          10271
CO            8220
CC(C)O        7298
O             6814
CCOC(C)=O     6802
CCCO          6616
CC(C)=O       6062
CCCCO         5613
CC#N          5251
CN(C)C=O      2767
Cc1ccccc1     2618
CC(C)CO       2440
C1COCCO1      2183
COC(C)=O      1908
C1CCOC1       1524


In [10]:
print('top 10 solutes:')
print(solute_counts.head(10).to_string())

top 10 solutes:
SMILES_Solute
CN1CCN(C2=Nc3cc(Cl)ccc3Nc3ccccc32)CC1                                                                   416
C[C@@H]1CC[C@@]2(OC1)O[C@H]1C[C@H]3[C@@H]4CC=C5C[C@@H](O)CC[C@]5(C)[C@H]4CC[C@]3(C)[C@H]1[C@@H]2C       380
Cn1c(=O)c2c(ncn2CC2OCCO2)n(C)c1=O                                                                       357
CC(=O)Nc1ccc(OC(=O)c2ccccc2OC(C)=O)cc1                                                                  313
O=C(O)c1ccccc1                                                                                          311
O=[N+]([O-])N1C2C3N([N+](=O)[O-])C1C1N([N+](=O)[O-])C(C(N1[N+](=O)[O-])N3[N+](=O)[O-])N2[N+](=O)[O-]    295
CC(C)C(=O)Nc1ccc([N+](=O)[O-])c(C(F)(F)F)c1                                                             272
Nc1ccc(S(N)(=O)=O)cc1                                                                                   267
O=c1[nH]cc(F)c(=O)[nH]1                                                                                 25